# Cardiovascular Disease Dataset - Data Preprocessing & Cleaning
**Course:** Machine Learning  
**Guide Implementation:** Data Cleaning, Handling Outliers, Visualization, Feature Scaling & Export

### Step 1. Import necessary libraries and load the raw dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load raw dataset
df = pd.read_csv('cardio_train.csv/cardio_train.csv', sep=';')
print("Initial Dataset Shape:", df.shape)
df.head()

### Step 2. Clean Column Names & Drop Unnecessary Identifiers

In [ ]:
# Strip spaces and ensure lowercase column names
df.columns = df.columns.str.strip().str.lower()

# Drop ID column as it has no predictive power
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)

print("Columns after cleanup:", list(df.columns))
df.head()

### Step 3. Fix Data Types & Feature Engineering (Age in Years & BMI)

In [ ]:
# Convert age from days to years
df['age'] = (df['age'] / 365.25).astype(int)

# Calculate Body Mass Index (BMI = weight / (height/100)^2)
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)
df['bmi'] = df['bmi'].round(2)

df[['age', 'height', 'weight', 'bmi']].head()

### Step 4. Handle Missing Values & Remove Duplicate Rows

In [ ]:
# Check missing values
print("Missing Values:\n", df.isnull().sum())

# Check and drop duplicate rows
duplicates_count = df.duplicated().sum()
print(f"Found {duplicates_count} duplicate rows.")
df.drop_duplicates(inplace=True)
print("Dataset Shape after dropping duplicates:", df.shape)

### Step 5. Detect and Clean Outliers (Physiological Ranges)

In [ ]:
# Check summary statistics before outlier treatment
print("Summary stats before outlier removal:")
display(df[['height', 'weight', 'ap_hi', 'ap_lo', 'bmi']].describe())

# Filtering invalid physiological ranges:
# 1. Height between 100 cm and 220 cm
# 2. Weight between 30 kg and 200 kg
# 3. Systolic BP (ap_hi) between 60 mmHg and 240 mmHg
# 4. Diastolic BP (ap_lo) between 40 mmHg and 180 mmHg
# 5. Systolic BP must be strictly greater than Diastolic BP
df = df[(df['height'] >= 100) & (df['height'] <= 220)]
df = df[(df['weight'] >= 30) & (df['weight'] <= 200)]
df = df[(df['ap_hi'] >= 60) & (df['ap_hi'] <= 240)]
df = df[(df['ap_lo'] >= 40) & (df['ap_lo'] <= 180)]
df = df[df['ap_hi'] > df['ap_lo']]

print("Dataset Shape after outlier removal:", df.shape)
display(df[['height', 'weight', 'ap_hi', 'ap_lo', 'bmi']].describe())

### Step 6. Visualize Data Distributions & Categorical Variables

In [ ]:
# Set plot style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age Distribution
sns.histplot(df['age'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Age Distribution (Years)')

# Systolic BP Boxplot
sns.boxplot(x=df['ap_hi'], ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('Systolic BP (ap_hi) Distribution')

# BMI Distribution
sns.histplot(df['bmi'], kde=True, ax=axes[1, 0], color='salmon')
axes[1, 0].set_title('BMI Distribution')

# Target Class Count Plot (Cardiovascular Disease)
sns.countplot(x='cardio', data=df, ax=axes[1, 1], hue='cardio', palette='Set2', legend=False)
axes[1, 1].set_title('Cardiovascular Disease Target Count (0 = No, 1 = Yes)')

plt.tight_layout()
plt.show()

### Step 7. Scale Numerical Features using MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
num_cols = ['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi']

# Create scaled version of dataset for ML models
df_scaled = df.copy()
df_scaled[num_cols] = scaler.fit_transform(df[num_cols])

print("Scaled Numerical Features sample:")
df_scaled[num_cols].head()

### Step 8. Export Cleaned & Processed Datasets

In [ ]:
# Save unscaled cleaned dataset
df.to_csv('cardio_cleaned.csv', index=False)
print("Saved unscaled cleaned dataset to: cardio_cleaned.csv")

# Save scaled cleaned dataset
df_scaled.to_csv('cardio_cleaned_scaled.csv', index=False)
print("Saved scaled cleaned dataset to: cardio_cleaned_scaled.csv")